# 4B · The Finance Exhibits
### Financial Analytics — Module 4

Four charts that exist *specifically* because of finance, each built from scratch so you own it:

1. **Waterfall (bridge)** — how we got from revenue to profit, step by step
2. **Candlestick** — a trading day in one glyph
3. **Correlation heatmap** — the diversification map
4. **Tornado** — which assumption matters most (the bridge to Module 10)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

BASE = "data/"
fin = pd.read_csv(BASE + "company_financials.csv")
px  = pd.read_csv(BASE + "nifty50_prices.csv", parse_dates=["date"])
uni = pd.read_csv(BASE + "nse_stock_universe.csv", parse_dates=["date"])
GREEN, RED, BLUE, GREY = "#16A34A", "#DC2626", "#2563EB", "#94A3B8" 

---
## 1. The Waterfall — from revenue to profit

The CFO's favourite chart: start at revenue, subtract each cost block, land on profit. Every step visible, nothing hidden. We build it manually — a waterfall is just bars with moving baselines, and building it once means you can customise it forever.

In [ ]:
yr = fin[fin["fiscal_year"] == "FY25-26"].iloc[0]

steps = [
    ("Revenue",        yr["revenue_cr"],        "start"),
    ("COGS",          -yr["cogs_cr"],           "down"),
    ("Employees",     -yr["employee_cost_cr"],  "down"),
    ("Marketing",     -yr["marketing_cr"],      "down"),
    ("Other opex",    -yr["other_opex_cr"],     "down"),
    ("Depreciation",  -yr["depreciation_cr"],   "down"),
    ("Interest",      -yr["interest_cr"],       "down"),
    ("Tax",           -yr["tax_cr"],            "down"),
    ("PAT",            yr["pat_cr"],            "end"),
]

fig, ax = plt.subplots(figsize=(11, 5))
running = 0
for i, (label, val, kind) in enumerate(steps):
    if kind == "start":
        ax.bar(i, val, bottom=0, color=BLUE); running = val
    elif kind == "end":
        ax.bar(i, val, bottom=0, color=GREEN)
    else:
        ax.bar(i, val, bottom=running, color=RED, alpha=0.85)   # negative bar hangs DOWN from running
        running += val
    top = running if kind != "end" else val
    ax.text(i, top + 200, f"{abs(val):,.0f}", ha="center", fontsize=8.5)

ax.set_xticks(range(len(steps)), [s[0] for s in steps], rotation=30, ha="right")
ax.set_title(f"MoneyMart FY25-26: Rs {yr['revenue_cr']:,.0f} cr of revenue became Rs {yr['pat_cr']:,.0f} cr of profit",
             loc="left", fontweight="bold")
ax.set_ylabel("Rs crore")
plt.tight_layout(); plt.show()

# Sanity check - the bridge must reconcile (Module 1: auditability)
walked = yr["revenue_cr"] + sum(v for _, v, k in steps if k == "down")
print(f"Bridge lands on {walked:,.1f} vs reported PAT {yr['pat_cr']:,.1f}  -> reconciles: {abs(walked - yr['pat_cr']) < 1}")

**The reconciliation check at the end is not optional.** A waterfall that doesn't land exactly on the reported figure is a bug in your chart or a hole in the data — either way, find out *before* presenting.

### ✏️ Exercise 1
Build the same waterfall for **FY20-21** (the COVID year). Which cost block shrank the least when revenue fell? (That stickiness has a name — operating leverage — and Module 5 will quantify it.)

In [ ]:
# your code here


---
## 2. The Candlestick — a day in one glyph

Each candle encodes four numbers: the **body** spans open→close (green if close>open, red otherwise); the **wicks** stretch to high and low. Traders read hundreds at a glance. We draw ~60 sessions manually with matplotlib — no special library, full understanding.

In [ ]:
d = px.dropna(subset=["open"]).tail(60).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(11, 4.5))
for i, r in d.iterrows():
    color = GREEN if r["close"] >= r["open"] else RED
    ax.vlines(i, r["low"], r["high"], color=color, lw=0.8)                    # wick
    ax.bar(i, abs(r["close"] - r["open"]), bottom=min(r["open"], r["close"]),
           color=color, width=0.65)                                           # body

ticks = range(0, len(d), 10)
ax.set_xticks(ticks, [d.loc[i, "date"].strftime("%d %b") for i in ticks])
ax.set_title("NIFTY 50 — last 60 sessions, candlestick", loc="left", fontweight="bold")
plt.tight_layout(); plt.show()

Reading practice: long upper wicks = the day reached high and was sold back down; a cluster of small bodies = indecision. In Module 11 the *order book* explains what's happening inside each candle.

---
## 3. The Correlation Heatmap — the diversification map

The message: *"Which stocks move together?"* — the exhibit on which all of Module 12 stands. Returns first (never correlate prices — trending prices show fake correlation), then a colour-coded matrix.

In [ ]:
wide = uni.pivot(index="date", columns="ticker", values="close")
rets = wide.pct_change().dropna()
corr = rets.corr()

# Order columns by sector so the blocks become visible
order = uni.drop_duplicates("ticker").sort_values("sector")["ticker"].tolist()
corr = corr.loc[order, order]

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, cmap="RdBu_r", vmin=-0.2, vmax=0.8, center=0.25,
            square=True, cbar_kws={"shrink": 0.7}, ax=ax,
            xticklabels=[t.replace(".NS","") for t in corr.columns],
            yticklabels=[t.replace(".NS","") for t in corr.index])
ax.set_title("Return correlations, 20 NSE stocks (sector-ordered)", loc="left", fontweight="bold")
plt.tight_layout(); plt.show()

pairs = corr.where(np.triu(np.ones_like(corr, dtype=bool), 1))
print(f"Average pairwise correlation: {pairs.stack().mean():.2f}")
print("Most correlated pair :", pairs.stack().idxmax(), round(pairs.stack().max(), 2))
print("Least correlated pair:", pairs.stack().idxmin(), round(pairs.stack().min(), 2))

See the darker squares hugging the diagonal? Those are **sector blocks** — banks move with banks, IT with IT. Diversifying across five banks is barely diversifying. This one picture *is* the case for spreading across sectors, and Module 12's optimiser will exploit exactly this matrix.

### ✏️ Exercise 2
Compute each stock's average correlation with all others (`corr.mean()`), sort it. The lowest-correlation stock is the best *diversifier* in this universe. Which is it, and does its sector explain why?

In [ ]:
# your code here


---
## 4. The Tornado — which assumption matters most

The prescriptive-analytics preview. Take a simple valuation with three assumptions; wiggle each one ±20% while holding the others; draw the impact as horizontal bars, widest on top. The chart instantly answers: *where should we spend our worrying?*

In [ ]:
def quick_value(revenue_growth=0.12, margin=0.085, multiple=18):
    """Toy valuation: next year's profit x an earnings multiple (Rs crore)."""
    rev_next = fin["revenue_cr"].iloc[-1] * (1 + revenue_growth)
    return rev_next * margin * multiple

base = quick_value()
assumptions = {"Revenue growth": "revenue_growth", "Profit margin": "margin", "Earnings multiple": "multiple"}
defaults = {"revenue_growth": 0.12, "margin": 0.085, "multiple": 18}

lows, highs, labels = [], [], []
for label, arg in assumptions.items():
    lo = quick_value(**{**defaults, arg: defaults[arg] * 0.8})
    hi = quick_value(**{**defaults, arg: defaults[arg] * 1.2})
    lows.append(lo - base); highs.append(hi - base); labels.append(label)

order = np.argsort(np.array(highs) - np.array(lows))
fig, ax = plt.subplots(figsize=(9, 3.5))
for j, i in enumerate(order):
    ax.barh(j, highs[i], color="#16A34A", alpha=0.8)
    ax.barh(j, lows[i],  color="#DC2626", alpha=0.8)
ax.set_yticks(range(len(order)), [labels[i] for i in order])
ax.axvline(0, color="black", lw=0.8)
ax.set_title(f"Tornado: value impact of ±20% in each assumption (base Rs {base:,.0f} cr)",
             loc="left", fontweight="bold")
ax.set_xlabel("Change in valuation (Rs crore)")
plt.tight_layout(); plt.show()

Every bar the same ±20% wiggle — yet the impacts differ hugely. That asymmetry of *sensitivity* is the whole message, and it tells the analyst where precision is worth buying. In Module 10 the wiggles become full probability distributions and the tornado becomes a Monte Carlo.

### ✏️ Exercise 3
Add a fourth assumption to `quick_value` — a cost inflation factor that reduces margin — and rebuild the tornado. Does it beat "earnings multiple" for the top spot?

---
## Recap
Waterfall = the P&L as a journey (and it must reconcile). Candlestick = four numbers, one glyph. Heatmap = the diversification map with sector blocks. Tornado = ranked sensitivity. **Next: turning charts into a living dashboard — Streamlit.**

*AI disclosure: ______*

In [ ]:
# workspace
